In [8]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point, LineString
import os
import numpy as np


In [9]:
import pickle
data_path = "../../data/hcm_data/"
train_path = os.path.join(data_path, "train.npy")
data = np.load(train_path, allow_pickle=True)
print(f"Loaded data from {train_path}, shape: {data.shape}, dtype: {data.dtype}, sample element: {data[0]}")

Loaded data from ../../data/hcm_data/train.npy, shape: (765036, 6), dtype: object, sample element: [19497
 list([21264, 4999, 21263, 26818, 5309, 43114, 25057, 5690, 43455, 21314, 25072, 25074, 21302, 25133, 21299, 12074, 42247, 42248, 42250, 42252, 5449, 18145, 21309, 44724, 25823, 1194, 21863, 21857, 21368, 21257, 22187, 45642, 22166, 26287, 39457, 26277, 3565, 4710, 14057, 25818, 4433, 21258, 44439, 11330, 11329, 3909, 10039, 10037, 9454, 32938, 10040, 17554, 532, 46138, 40651, 45702, 12573, 45257, 12577, 45247, 12579, 45238, 45233, 12571, 12563, 12568, 40410, 12566, 17982, 45216, 45250, 2074, 9655, 45262, 9649, 10118, 10068, 40374, 26662, 4562, 26207, 13916, 13915, 17508, 40379, 27623, 22521, 22517, 23062, 16843, 23057, 23063, 23068, 23069, 23074, 22512, 22514, 22531, 16969, 22530, 22536, 22535, 28001, 27374, 12220, 4058, 16802, 11411, 11410, 31626, 12222, 11406, 16793, 30095, 32413, 29987, 30102, 44166, 32415, 20886, 23573, 871, 23570, 29255, 14204, 29243, 2543, 23095, 4690, 28536

In [10]:
N = len(data)
sample_size = int(0.1 * N)

rng = np.random.default_rng(42)  # reproducible
indices = rng.choice(N, size=sample_size, replace=False)

sample = data[indices]
print(sample.shape)

(76503, 6)


In [13]:
with open(os.path.join(data_path,"nwk_hcm/hcm_edges_poi_new_simplify.pkl"), 'rb') as f:
    edgeinfo = pickle.load(f)
    
sample_edge = iter(edgeinfo.values()).__next__()
print("Full edge list  :", sample_edge)
print("Total length    :", len(sample_edge))

Full edge list  : ['residential', 29.493367082668353, '366367223', '3771499617', 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Total length    : 14


In [19]:
import numpy as np

def poi_analysis_normalized(data, edgeinfo):
    poi_density = []
    times = []

    for d in data:
        edge_list = d[1]
        time = d[-1]

        total_len = 0.0
        total_poi = None

        for eid in edge_list:
            if eid not in edgeinfo:
                continue

            edge = edgeinfo[eid]
            poi_vec = np.array(edge[5:], dtype=np.float32)  # force float

            if total_poi is None:
                total_poi = np.zeros_like(poi_vec, dtype=np.float32)

            total_poi += poi_vec
            total_len += float(edge[1])

        if total_poi is None or total_len == 0:
            continue

        density = total_poi / (total_len + 1e-6)

        poi_density.append(density)
        times.append(time)

    return np.array(poi_density), np.array(times)

In [20]:
poi, time = poi_analysis_normalized(data, edgeinfo)

corrs = []
for i in range(poi.shape[1]):
    c = np.corrcoef(poi[:, i], time)[0, 1]
    corrs.append(c)

print(corrs)

c:\Users\atom0\miniconda3\envs\ml\lib\site-packages\numpy\lib\function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\atom0\miniconda3\envs\ml\lib\site-packages\numpy\lib\function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


[nan, -0.09666906900942042, -0.013046787902169569, -0.06979601580528033, -0.10093089642149916, -0.010544661821967475, -0.00859644925547503, -0.008633288029603201, 0.06289052574739211]
